# Histogram Gradient Boosted Trees

## Introduction

After evaluating both linear models and tree-based ensemble methods such as decision trees and random forests, we will explore boosting-based approaches. While random forests improve predictive performance by reducing variance through the aggregation of many independent trees, gradient boosting follows a fundamentally different strategy: it builds models sequentially, with each new tree focusing on correcting the errors made by the previous ones.

This sequential learning process allows gradient boosting to capture complex nonlinear relationships and feature interactions more effectively than single trees or bagging-based methods. As a result, boosting models are often among the strongest performers on structured and tabular datasets, particularly in problems such as customer churn prediction, where patterns are rarely purely linear.

In this chapter, the focus is on the `HistGradientBoostingClassifier` from `scikit-learn`. This implementation introduces a histogram-based optimization, where continuous features are discretized into bins before training. This significantly improves computational efficiency while maintaining strong predictive performance, making it well-suited for medium to large datasets.

## Table of Contents

1. [Modeling Considerations](#modeling-considerations)
2. [Model Definition and Hyperparameter Tuning](#model-definition-and-hyperparameter-tuning)
3. [Model Implementation](#model-implementation)
   - [Initial Grid Search](#initial-grid-definition-and-cross-validation)
   - [Refining Grid Search](#refined-hyperapameter-grid)
   - [Hyperparameter Tuning Results](#refined-grid-search-results)
   - [Model Interpretability](#model-interpretability)
4. [Test Set Evaluation](#test-set-evaluation)
5. [Executive Summary](#executive-summary)

## Modeling Considerations

Before fitting the gradient boosting model, several key considerations are outlined that influence model behavior, evaluation, and generalization performance.

---

### Class Imbalance

The target variable exhibits moderate class imbalance, with approximately 26.5% of customers churning.

Gradient boosting models optimize a differentiable loss function and iteratively focus on correcting previous errors. While this can improve the model’s ability to identify minority-class observations, it does not inherently resolve class imbalance. In some cases, the model may still prioritize improvements on the majority class, especially when minimizing overall loss.

As in previous chapters, accuracy is not used as a primary evaluation metric. Instead, emphasis is placed on recall, precision, F1 score, and PR-AUC, which provide a more informative assessment of performance on the minority class. Additionally, threshold tuning is incorporated as a key step to balance business-relevant trade-offs between correctly identifying churners and limiting false positives.

---

### Feature Scaling

Histogram-based gradient boosting models do not require feature scaling. Similar to decision trees, splits are determined based on feature thresholds rather than distance-based calculations, making the scale of input variables largely irrelevant.

Furthermore, the histogram-based approach discretizes continuous features into bins internally, reducing sensitivity to feature magnitude and improving computational efficiency. As a result, all features are used in their original form, maintaining consistency with the existing data preparation pipeline.

---

### Feature Representation

Gradient boosting models, like decision trees and random forests, naturally capture non-linear relationships and interaction effects. This reduces the need for explicit feature transformations or manually engineered interaction terms.

Continuous variables are retained in their original form, allowing the model to learn optimal split points during training. Manual discretization is not required and may in fact reduce model performance by removing useful information.

Categorical variables are represented using the same encoded structure as in previous tree-based models, ensuring consistency across experiments and enabling direct comparison of model performance.

---

### Model Complexity and Overfitting

Gradient boosting models are highly flexible and can achieve strong predictive performance, but this flexibility comes with a **significant risk of overfitting**.

Unlike random forests, which reduce variance through averaging independent trees, boosting builds trees sequentially. Each new tree is explicitly trained to correct the errors of the existing model, which can lead to progressively fitting noise in the training data if the model is not properly regularized.

Overfitting risk is particularly influenced by:

* the number of boosting iterations
* the learning rate
* tree depth and structure
* minimum samples per leaf

To mitigate this, model complexity is carefully controlled through hyperparameters such as learning rate, maximum depth, minimum samples per leaf, and L2 regularization. In practice, smaller learning rates combined with a controlled number of iterations are often preferred to achieve better generalization.

---

### Sequential Learning and Parameter Sensitivity

A key distinction of gradient boosting is its **sequential learning process**, where each tree depends on the previous ones. This makes the model more sensitive to hyperparameter choices compared to bagging-based methods.

Small changes in parameters such as learning rate or tree depth can significantly affect model performance. As a result, hyperparameter tuning plays a critical role in achieving optimal results, and parameter interactions must be considered rather than evaluated in isolation.

This sensitivity reinforces the need for a structured and systematic tuning approach.

---

### Validation Strategy and Iterative Model Refinement

Hyperparameter tuning and model refinement are performed iteratively, which introduces a risk of overfitting to evaluation data if not properly controlled.

To address this, the training data is split into a training subset and a separate validation set. Cross-validation is applied within the training subset to evaluate candidate hyperparameter configurations, while the validation set is used for model comparison, refinement, and threshold selection.

This approach is particularly important for gradient boosting models, where repeated tuning can easily lead to overly optimistic performance estimates due to their high flexibility.

A final hold-out test set is reserved exclusively for unbiased evaluation and is not used during model development.

---

### Summary

Gradient boosting extends tree-based modeling by sequentially improving predictions and capturing complex patterns in the data. This often leads to stronger predictive performance compared to single trees or bagging-based ensembles.

However, this increased modeling power comes with higher sensitivity to hyperparameters and a greater risk of overfitting. The modeling approach therefore emphasizes careful regularization, structured hyperparameter tuning, and robust validation practices.

With these considerations in place, the gradient boosting model can be effectively developed and evaluated in a controlled and reproducible manner.


## Model Definition and Hyperparameter Tuning

The gradient boosting model is implemented using the `HistGradientBoostingClassifier` from scikit-learn. Unlike decision trees, gradient boosting builds an ensemble of trees sequentially, where each new tree refines the predictions of the existing model.

Model behavior is controlled through hyperparameters that govern both the complexity of individual trees and the overall learning process. The most important parameters include the learning rate, the number of boosting iterations (`max_iter`), tree depth (`max_depth`), minimum samples per leaf, and L2 regularization. In particular, the interaction between learning rate and number of iterations plays a central role in determining model performance.

---

### Hyperparameter Tuning Strategy

To identify an appropriate configuration, hyperparameters are explored using grid search with cross-validation. This allows multiple parameter combinations to be evaluated across different data splits, providing a stable basis for model comparison.

Given the model’s sensitivity to parameter choices, tuning is performed in a structured manner, focusing on a reasonable range of values rather than an exhaustive search.

---

### Evaluation Metric

Due to class imbalance, model comparison is based on **PR-AUC (average precision)**. This metric emphasizes the model’s ability to correctly rank churners and is more informative than accuracy for this task.

---

### Model Selection Framework

Cross-validation is used to identify promising hyperparameter combinations, but final model selection is based on performance on a separate validation set. This helps ensure that the chosen configuration generalizes beyond the data used during tuning.

The final model will subsequently be evaluated on a hold-out test set to obtain an unbiased estimate of performance.

---

In the following section, we define the hyperparameter grid and perform cross-validated model exploration.


We will begin by loading data and performing the train vlaidation split:

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split

# Load datasets
train_df = pl.read_csv("./data/processed/06_dpp_train_df.csv")
test_df = pl.read_csv("./data/processed/06_dpp_test_df.csv")

selected_cols = [
    'SeniorCitizenRelevel',
    'Partner',
    'Dependents',
    'tenure',
    'Contract',
    'PaperlessBilling',
    'MonthlyCharges',
    'Churn',
    'AdditionalInternetServicesCount',
    'StreamingServicesCount',
    'PaymentMethod_bin_3'
]

train_df = train_df.select(selected_cols)
test_df = test_df.select(selected_cols)

# Convert to pandas for sklearn
train_pd = train_df.to_pandas()

# Stratified split: 25% of current train -> validation
train_sub_pd, val_pd = train_test_split(
    train_pd,
    test_size=0.25,
    stratify=train_pd["Churn"],
    random_state=42
)

# Back to polars
train_sub_df = pl.from_pandas(train_sub_pd)
val_df = pl.from_pandas(val_pd)

# Shapes
print("Train subset shape:", train_sub_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Class proportions helper
def class_proportions(df, name):
    print(f"\n{name} class proportions:")
    print(
        df.group_by("Churn")
        .len()
        .with_columns(
            (pl.col("len") / pl.col("len").sum()).alias("proportion")
        )
        .sort("Churn")
    )

class_proportions(train_sub_df, "Train subset")
class_proportions(val_df, "Validation")
class_proportions(test_df, "Test")

Train subset shape: (4225, 11)
Validation shape: (1409, 11)
Test shape: (1409, 11)

Train subset class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 3104 ┆ 0.734675   │
│ Yes   ┆ 1121 ┆ 0.265325   │
└───────┴──────┴────────────┘

Validation class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘

Test class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘


preparing data for modelling:

In [2]:
from src.utils.data_preparation_models import prepare_tree_features

In [ ]:
categorical_cols = ['SeniorCitizenRelevel',
                    'Partner',
                    'Dependents',
                    'Contract',
                    'PaperlessBilling',
                    'PaymentMethod_bin_3']

target_col = "Churn"

numerical_cols = ['tenure', 
                  'MonthlyCharges', 
                  'AdditionalInternetServicesCount', 
                  'StreamingServicesCount']   

categorical_orders = { "SeniorCitizenRelevel": ["No", "Yes"], 
                      "Partner": ["No", "Yes"], 
                      "Dependents": ["No", "Yes"], 
                      "PaperlessBilling": ["No", "Yes"], 
                      "Contract": ["Month-to-month", "One year", "Two year"] }

train_X, train_y = prepare_tree_features(
    df=train_sub_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col
)

validation_X, validation_y = prepare_tree_features(
    df=val_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

In [4]:
print(train_X.shape)
print(validation_X.shape)

(4225, 18)
(1409, 18)


Categorical variables are one-hot encoded to provide a fully numeric input matrix for the model.

Some features created during feature engineering are represented as short-range integer variables. These are retained as ordinal numeric features rather than being one-hot encoded. Preserving their ordering allows the boosting model to learn effective threshold-based splits across successive trees.

The resulting design matrix therefore consists of a combination of one-hot encoded categorical variables, ordinal integer features, and continuous variables. This representation aligns naturally with gradient boosting, enabling the model to capture non-linear relationships and interaction effects through sequential tree-based refinements.

### Initial Grid Definition and Cross-Validation

The gradient boosting model is initially explored using cross-validated grid search to identify promising regions of the hyperparameter space. This approach evaluates multiple configurations across different data splits within the training subset, providing a stable estimate of relative model performance.

Model comparison during this stage is based on PR-AUC (average precision), which is well-suited for imbalanced classification problems and emphasizes the model’s ability to correctly rank and identify churners.

The hyperparameter grid is designed to capture key aspects of the boosting process, with particular focus on learning rate, number of iterations, and tree depth, along with regularization parameters such as minimum samples per leaf and L2 regularization. These parameters jointly control how the model learns over successive trees and how complexity is managed.

At this stage, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate configurations identified through cross-validation will be further evaluated on a separate validation set to guide final model selection.

The tuning process therefore aims to identify configurations that provide strong predictive performance while remaining stable across validation folds.


In [5]:
param_grid = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_iter": [100, 200, 300],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [10, 20, 40],
    "l2_regularization": [0.0, 0.1, 1.0]
}

### Hyperparameter Grid Design

The hyperparameter grid is designed to explore key aspects of the gradient boosting process, balancing model flexibility with regularization. Each parameter controls a different component of how the model learns from the data.

---

#### `learning_rate`

The learning rate controls how much each individual tree contributes to the overall model.

* Smaller values result in more gradual learning, where each tree makes only a small adjustment
* Larger values allow the model to learn faster but increase the risk of overfitting

In practice, lower learning rates often lead to better generalization but require more boosting iterations to achieve strong performance.

---

#### `max_iter`

This parameter defines the number of boosting iterations, i.e., the number of trees added sequentially to the model.

* More iterations increase model capacity and allow the model to capture more complex patterns
* However, too many iterations can lead to overfitting, especially when combined with a high learning rate

This parameter works closely with the learning rate and is typically interpreted in combination with it.

---

#### `max_depth`

The maximum depth controls the complexity of individual trees within the ensemble.

* Shallow trees (e.g., depth 3) act as weak learners and focus on simple patterns
* Deeper trees can capture more complex interactions but increase the risk of overfitting

Limiting tree depth is one of the primary ways to regularize gradient boosting models.

---

#### `min_samples_leaf`

This parameter specifies the minimum number of samples required in each leaf node.

* Larger values prevent the model from creating very small, highly specific splits
* This acts as a regularization mechanism, improving model stability and reducing sensitivity to noise

It is particularly important in boosting, where successive trees may otherwise overfit residual patterns.

---

#### `l2_regularization`

L2 regularization penalizes overly complex models by discouraging large adjustments in predictions.

* Higher values increase regularization strength and help prevent overfitting
* Lower values allow the model more flexibility to fit the data

This parameter provides an additional layer of control beyond tree structure.

---

### Summary

Together, these hyperparameters control how the model learns over successive iterations, how complex individual trees can become, and how strongly the model is regularized.

The grid is designed to explore different trade-offs between learning speed, model complexity, and generalization performance, allowing the identification of configurations that perform well across validation folds.


Run cross validation on grid search

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

hgb = HistGradientBoostingClassifier(random_state=42)

grid = GridSearchCV(
    estimator=hgb,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid.fit(train_X, train_y)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'l2_regularization': [0.0, 0.1, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [3, 5, ...], 'max_iter': [100, 200, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time f

Extract the best candidates to validate

In [12]:
import pandas as pd

cv_results = pd.DataFrame(grid.cv_results_)

cv_summary = (
    cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

cv_summary.head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.667472,0.052243,0.725302,0.013286,"{'l2_regularization': 0.1, 'learning_rate': 0...."
1,2,0.667298,0.050727,0.725592,0.013671,"{'l2_regularization': 0.0, 'learning_rate': 0...."
2,3,0.667112,0.049966,0.711543,0.013473,"{'l2_regularization': 0.0, 'learning_rate': 0...."
3,4,0.667073,0.052740,0.728886,0.013707,"{'l2_regularization': 1.0, 'learning_rate': 0...."
4,5,0.666838,0.051272,0.733290,0.011592,"{'l2_regularization': 0.0, 'learning_rate': 0...."
5,6,0.666817,0.053956,0.738321,0.012256,"{'l2_regularization': 0.0, 'learning_rate': 0...."
6,7,0.666696,0.051162,0.726484,0.013521,"{'l2_regularization': 1.0, 'learning_rate': 0...."
7,8,0.666627,0.050804,0.711440,0.013736,"{'l2_regularization': 0.1, 'learning_rate': 0...."
8,9,0.666615,0.051809,0.732313,0.012302,"{'l2_regularization': 1.0, 'learning_rate': 0...."
9,10,0.666607,0.050869,0.721899,0.014130,"{'l2_regularization': 1.0, 'learning_rate': 0...."


In [13]:
top_candidates = cv_summary.head(10).copy()

top_candidates

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.667472,0.052243,0.725302,0.013286,"{'l2_regularization': 0.1, 'learning_rate': 0...."
1,2,0.667298,0.050727,0.725592,0.013671,"{'l2_regularization': 0.0, 'learning_rate': 0...."
2,3,0.667112,0.049966,0.711543,0.013473,"{'l2_regularization': 0.0, 'learning_rate': 0...."
3,4,0.667073,0.052740,0.728886,0.013707,"{'l2_regularization': 1.0, 'learning_rate': 0...."
4,5,0.666838,0.051272,0.733290,0.011592,"{'l2_regularization': 0.0, 'learning_rate': 0...."
5,6,0.666817,0.053956,0.738321,0.012256,"{'l2_regularization': 0.0, 'learning_rate': 0...."
6,7,0.666696,0.051162,0.726484,0.013521,"{'l2_regularization': 1.0, 'learning_rate': 0...."
7,8,0.666627,0.050804,0.711440,0.013736,"{'l2_regularization': 0.1, 'learning_rate': 0...."
8,9,0.666615,0.051809,0.732313,0.012302,"{'l2_regularization': 1.0, 'learning_rate': 0...."
9,10,0.666607,0.050869,0.721899,0.014130,"{'l2_regularization': 1.0, 'learning_rate': 0...."


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from src.utils.classification_metrics import compute_classification_metrics

import pandas as pd

def evaluate_hbt_candidates_on_validation(
    top_candidates,
    train_X,
    train_y,
    validation_X,
    validation_y,
    random_state=42,
    sort_by="pr_auc"
):
    """
    Fit histogram gradient boosted tree candidate models on training data and evaluate on validation set.

    Parameters
    ----------
    top_candidates : pd.DataFrame
        DataFrame containing at least:
        - 'params'
        - 'rank_test_score'
        - 'mean_test_score'
        - 'std_test_score'

    train_X, train_y : training data
    validation_X, validation_y : validation data

    random_state : int
    sort_by : metric to sort results by (default 'pr_auc')

    Returns
    -------
    pd.DataFrame
        Validation results sorted by chosen metric
    """

    candidate_results = []

    for _, row in top_candidates.iterrows():
        params = row["params"]

        model = HistGradientBoostingClassifier(
            **params,
            random_state=random_state
        )

        model.fit(train_X, train_y)

        val_prob = model.predict_proba(validation_X)[:, 1]
        val_pred = model.predict(validation_X)

        metrics = compute_classification_metrics(
            y_true=validation_y,
            y_pred=val_pred,
            y_prob=val_prob
        )

        candidate_results.append({
            "cv_rank": row["rank_test_score"],
            "cv_pr_auc": row["mean_test_score"],
            "cv_std": row["std_test_score"],
            "params": params,
            **metrics
        })

    results_df = (
        pd.DataFrame(candidate_results)
        .sort_values(by=sort_by, ascending=False)
        .reset_index(drop=True)
    )

    return results_df

In [15]:
candidate_results_df = evaluate_hbt_candidates_on_validation(
    top_candidates=top_candidates,
    train_X=train_X,
    train_y=train_y,
    validation_X=validation_X,
    validation_y=validation_y
)

candidate_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,4,0.667073,0.052740,"{'l2_regularization': 1.0, 'learning_rate': 0....",0.789922,0.638298,0.481283,0.548780,0.821456,0.618095,180,933,102,194
1,5,0.666838,0.051272,"{'l2_regularization': 0.0, 'learning_rate': 0....",0.791341,0.645985,0.473262,0.546296,0.822318,0.618081,177,938,97,197
2,6,0.666817,0.053956,"{'l2_regularization': 0.0, 'learning_rate': 0....",0.789212,0.637993,0.475936,0.545176,0.821821,0.617377,178,934,101,196
3,1,0.667472,0.052243,"{'l2_regularization': 0.1, 'learning_rate': 0....",0.794180,0.650000,0.486631,0.556575,0.823263,0.616622,182,937,98,192
4,7,0.666696,0.051162,"{'l2_regularization': 1.0, 'learning_rate': 0....",0.793471,0.647687,0.486631,0.555725,0.822971,0.616500,182,936,99,192
5,9,0.666615,0.051809,"{'l2_regularization': 1.0, 'learning_rate': 0....",0.789212,0.637993,0.475936,0.545176,0.821946,0.615780,178,934,101,196
6,8,0.666627,0.050804,"{'l2_regularization': 0.1, 'learning_rate': 0....",0.789922,0.643382,0.467914,0.541796,0.822770,0.615492,175,938,97,199
7,2,0.667298,0.050727,"{'l2_regularization': 0.0, 'learning_rate': 0....",0.793471,0.647687,0.486631,0.555725,0.822584,0.615243,182,936,99,192
8,3,0.667112,0.049966,"{'l2_regularization': 0.0, 'learning_rate': 0....",0.793471,0.653137,0.473262,0.548837,0.822679,0.615058,177,941,94,197
9,10,0.666607,0.050869,"{'l2_regularization': 1.0, 'learning_rate': 0....",0.794890,0.652330,0.486631,0.557427,0.822761,0.613703,182,938,97,192


### Validation Set Comparison and Model Selection

We select **model index 9** as the final model based on validation set performance.

Although model index 3 achieves the highest PR-AUC of **0.6166**, the selected model (index 9) achieves a very similar PR-AUC of **0.6137**, representing a difference of only **0.0029**. This difference is small and within the range of expected variation, and is therefore not considered practically significant.

At the same time, model index 9 demonstrates slightly stronger performance across other metrics:

* **F1 score**: 0.5574 (vs 0.5566 for model 3)
* **Precision**: 0.6523 (vs 0.6500 for model 3)
* **Recall**: identical at 0.4866

In addition, model index 9 shows slightly better stability in cross-validation, with a lower standard deviation of **0.0509** compared to **0.0522** for model index 3.

Given that recall remains unchanged and PR-AUC is nearly identical, the improvement in F1 score and precision, along with increased stability, indicates a more balanced and robust model.

For these reasons, model index 9 is selected as the final configuration, prioritizing overall performance consistency over a negligible difference in a single metric.


In [16]:
model_1_results = candidate_results_df.loc[9]
print("Model results:")
print(model_1_results)
model_1_params = model_1_results["params"]
print("Model Parameters:")
print(model_1_params)


Model results:
cv_rank                                                     10
cv_pr_auc                                             0.666607
cv_std                                                0.050869
params       {'l2_regularization': 1.0, 'learning_rate': 0....
accuracy                                               0.79489
precision                                              0.65233
recall                                                0.486631
f1                                                    0.557427
auc                                                   0.822761
pr_auc                                                0.613703
tp                                                         182
tn                                                         938
fp                                                          97
fn                                                         192
Name: 9, dtype: object
Model Parameters:
{'l2_regularization': 1.0, 'learning_rate': 0.03, 'max_depth':

### Refined Hyperapameter Grid

Based on the initial validation results, a refined hyperparameter grid is defined around the selected gradient boosting configuration (model index 9). Rather than repeating a broad search, the refinement step focuses on a narrower region of the hyperparameter space centered on this candidate model.

Importantly, several of the selected model’s parameters lie at the **edges of the initial grid**, including learning rate, number of iterations, minimum samples per leaf, and L2 regularization. This suggests that the optimal configuration may lie slightly beyond the originally explored range. The refined grid therefore extends these parameters in the corresponding directions, allowing for smaller learning rates, higher iteration counts, larger leaf sizes, and stronger regularization.

At the same time, the refinement maintains a focus on relatively shallow trees, as the initial results consistently favored lower tree depth, indicating that simpler base learners generalize better in the boosting framework for this dataset.

This targeted search aims to determine whether incremental adjustments beyond the original grid boundaries lead to further improvements in validation performance, while preserving the stability and generalization characteristics observed in the initial model selection.


In [18]:
param_grid_refined = {
    "learning_rate": [0.01, 0.02, 0.03],
    "max_iter": [300, 400],
    "max_depth": [2, 3, 4],
    "min_samples_leaf": [30, 40, 50],
    "l2_regularization": [1.0, 2.0, 5.0]
}

In [19]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

hgb = HistGradientBoostingClassifier(random_state=42)

grid_refined = GridSearchCV(
    estimator=hgb,
    param_grid=param_grid_refined,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_refined.fit(train_X, train_y)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'l2_regularization': [1.0, 2.0, ...], 'learning_rate': [0.01, 0.02, ...], 'max_depth': [2, 3, ...], 'max_iter': [300, 400], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for ea

In [21]:
import pandas as pd

cv_refined_results = pd.DataFrame(grid_refined.cv_results_)

cv_refined_summary = (
    cv_refined_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

cv_refined_summary.head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.667848,0.050901,0.714549,0.012801,"{'l2_regularization': 5.0, 'learning_rate': 0...."
1,2,0.667720,0.052877,0.721999,0.012140,"{'l2_regularization': 2.0, 'learning_rate': 0...."
2,3,0.667556,0.052813,0.715901,0.013619,"{'l2_regularization': 5.0, 'learning_rate': 0...."
3,4,0.667498,0.049656,0.715225,0.013345,"{'l2_regularization': 5.0, 'learning_rate': 0...."
4,5,0.667436,0.051933,0.717732,0.013266,"{'l2_regularization': 2.0, 'learning_rate': 0...."
5,6,0.667412,0.051854,0.719970,0.013952,"{'l2_regularization': 2.0, 'learning_rate': 0...."
6,7,0.667406,0.050170,0.712012,0.013261,"{'l2_regularization': 5.0, 'learning_rate': 0...."
7,8,0.667275,0.050570,0.724459,0.012568,"{'l2_regularization': 5.0, 'learning_rate': 0...."
8,9,0.667244,0.051937,0.712285,0.013594,"{'l2_regularization': 5.0, 'learning_rate': 0...."
9,10,0.667174,0.052205,0.723853,0.012884,"{'l2_regularization': 1.0, 'learning_rate': 0...."


In [22]:
top_refined_candidates = cv_summary.head(10).copy()

In [23]:
candidate_refined_results_df = evaluate_hbt_candidates_on_validation(
    top_candidates=top_refined_candidates,
    train_X=train_X,
    train_y=train_y,
    validation_X=validation_X,
    validation_y=validation_y
)

candidate_refined_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,8,0.667275,0.050570,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.793471,0.647687,0.486631,0.555725,0.823332,0.617050,182,936,99,192
1,1,0.667848,0.050901,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.793471,0.648746,0.483957,0.554364,0.823593,0.616524,181,937,98,193
2,7,0.667406,0.050170,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.795600,0.654676,0.486631,0.558282,0.823850,0.615184,182,939,96,192
3,4,0.667498,0.049656,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.792051,0.647273,0.475936,0.548536,0.823454,0.614932,178,938,97,196
4,6,0.667412,0.051854,"{'l2_regularization': 2.0, 'learning_rate': 0....",0.792761,0.647482,0.481283,0.552147,0.822915,0.614860,180,937,98,194
5,5,0.667436,0.051933,"{'l2_regularization': 2.0, 'learning_rate': 0....",0.793471,0.649819,0.481283,0.552995,0.822952,0.614755,180,938,97,194
6,9,0.667244,0.051937,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.794890,0.652330,0.486631,0.557427,0.823577,0.614537,182,938,97,192
7,3,0.667556,0.052813,"{'l2_regularization': 5.0, 'learning_rate': 0....",0.789922,0.641304,0.473262,0.544615,0.823174,0.614472,177,936,99,197
8,2,0.667720,0.052877,"{'l2_regularization': 2.0, 'learning_rate': 0....",0.793471,0.649819,0.481283,0.552995,0.822477,0.613344,180,938,97,194
9,10,0.667174,0.052205,"{'l2_regularization': 1.0, 'learning_rate': 0....",0.791341,0.642857,0.481283,0.550459,0.821904,0.613268,180,935,100,194


In [24]:
model_1_results

cv_rank                                                     10
cv_pr_auc                                             0.666607
cv_std                                                0.050869
params       {'l2_regularization': 1.0, 'learning_rate': 0....
accuracy                                               0.79489
precision                                              0.65233
recall                                                0.486631
f1                                                    0.557427
auc                                                   0.822761
pr_auc                                                0.613703
tp                                                         182
tn                                                         938
fp                                                          97
fn                                                         192
Name: 9, dtype: object

### Refined Grid Search Results

The refined hyperparameter search was conducted around the previously selected model to evaluate whether extending the parameter space beyond the initial grid boundaries would lead to further improvements in performance.

While the refined grid explored lower learning rates, higher iteration counts, and stronger regularization, the resulting models showed only marginal differences in validation performance. The best configurations achieved PR-AUC values in the range of approximately 0.614–0.617, representing an improvement of less than 0.003 compared to the previously selected model.

Other evaluation metrics, including recall, precision, and F1 score, remained largely unchanged across the top candidates. In particular, recall — a key metric for churn prediction — showed no meaningful improvement.

These results indicate that the model has reached a point of diminishing returns with respect to hyperparameter tuning. The initial configuration already captures the majority of the predictive signal in the data, and further refinement yields only minor variations within expected statistical noise.

Given the negligible performance gains and to avoid overfitting to the validation set through excessive tuning, the previously selected model is retained as the final configuration.

This outcome highlights an important aspect of model development: beyond a certain point, improvements are better achieved through feature engineering or alternative modeling approaches rather than continued hyperparameter optimization.

### Training the best model on whole train dataset

In [25]:
train_val_X = pd.concat([train_X, validation_X], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

from sklearn.ensemble import HistGradientBoostingClassifier

final_model = HistGradientBoostingClassifier(
    **model_1_params,
    random_state=42,
)

final_model.fit(train_val_X, train_val_y)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.03
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",300
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",3
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",40
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dtyp

### Model Interpretability

Unlike a single decision tree, gradient boosting models do not provide a straightforward or easily interpretable set of decision rules. The final prediction is the result of many trees built sequentially, where each tree contributes a small adjustment to the overall model. As a result, individual decision paths are not directly accessible, and local interpretability is limited.

To gain insight into model behavior, global feature importance is used. Feature importance reflects the contribution of each variable to improving model performance across all boosting iterations. This provides a high-level view of which features are most influential in predicting churn.

### Feature Imporatnce

In [27]:
from sklearn.inspection import permutation_importance
import pandas as pd

perm = permutation_importance(
    final_model,
    validation_X,
    validation_y,
    n_repeats=20,
    random_state=42,
    scoring="average_precision"
)

feature_importance = pd.DataFrame({
    "feature": validation_X.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

feature_importance

,feature,importance_mean,importance_std
14,tenure,0.159265,0.015388
15,MonthlyCharges,0.132262,0.017479
6,Contract_Month-to-month,0.101099,0.015518
16,AdditionalInternetServicesCount,0.031709,0.010091
12,PaymentMethod_bin_3_Electronic check,0.011499,0.006419
9,PaperlessBilling_No,0.005686,0.006475
17,StreamingServicesCount,0.004205,0.001201
8,Contract_Two year,0.003556,0.003443
0,SeniorCitizenRelevel_No,0.003485,0.003310
7,Contract_One year,0.003336,0.000781


Permutation importance was used to assess the contribution of individual features to model performance. The results indicate that predictive power is highly concentrated in a small subset of variables.

The most important features are:

tenure (importance: 0.159)
MonthlyCharges (importance: 0.132)
Contract (Month-to-month) (importance: 0.101)

These features contribute substantially more than all others, suggesting that customer longevity, pricing, and contract type are the primary drivers of churn prediction in the model.

A second tier of features provides moderate additional signal:

AdditionalInternetServicesCount (0.032)
PaymentMethod (Electronic check) (0.011)

These variables refine the model’s predictions but have a significantly smaller impact compared to the top features.

All remaining features have very low importance values, many close to zero. This indicates that they contribute little to predictive performance and are rarely used in meaningful splits across the boosting process.

Some features even show slightly negative importance values, which can occur due to statistical variation or feature redundancy. This suggests that these variables do not provide independent predictive signal and may overlap with more informative features.

Overall, the results show that the model relies heavily on a small number of key variables, while many engineered or encoded features play only a marginal role. This concentration of importance is typical for boosting models, which tend to focus strongly on the most informative patterns in the data.

While permutation importance provides useful global insights, it is important to note that correlated features may share importance, and individual contributions should not be interpreted in isolation.

In [28]:
### Store the model for the consistency

import pickle

model_package = {
    # model
    "model": final_model,
    # data preparation setup
    "categorical_cols":categorical_cols,
    "numerical_cols":numerical_cols,
    "categorical_orders":categorical_orders,
    "target_col":target_col,
    "reference_columns": train_X.columns.tolist()
}

with open("./models/hist_boosted_trees_final.pkl", "wb") as f:
    pickle.dump(model_package, f)

## Test Set Evaluation

After completing model selection and hyperparameter refinement, the final gradient boosting model is evaluated on the hold-out test set.

The test set has not been used at any stage of model training, cross-validation, or validation-based model selection. As a result, it provides an unbiased estimate of the model’s generalization performance on unseen data.

The final model is retrained on the combined training and validation datasets to ensure that all available information is utilized. This improves parameter estimation while preserving the independence of the test set for evaluation.

Model performance is assessed using the same metrics applied throughout the modeling process, including ROC-AUC, PR-AUC, and threshold-based classification metrics such as precision, recall, and F1 score. This ensures consistency and allows direct comparison with previously developed models.

The results reported in this section represent the final estimate of model performance and form the basis for evaluating the effectiveness of the gradient boosting approach relative to earlier models.


In [29]:
from src.utils.classification_metrics import compute_classification_metrics

categorical_cols = ['SeniorCitizenRelevel',
                    'Partner',
                    'Dependents',
                    'Contract',
                    'PaperlessBilling',
                    'PaymentMethod_bin_3']

target_col = "Churn"

numerical_cols = ['tenure', 
                  'MonthlyCharges', 
                  'AdditionalInternetServicesCount', 
                  'StreamingServicesCount']   

categorical_orders = { "SeniorCitizenRelevel": ["No", "Yes"], 
                      "Partner": ["No", "Yes"], 
                      "Dependents": ["No", "Yes"], 
                      "PaperlessBilling": ["No", "Yes"], 
                      "Contract": ["Month-to-month", "One year", "Two year"] }

test_X_reduced, test_y_reduced = prepare_tree_features(
    df=test_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

In [30]:
test_probs = final_model.predict_proba(test_X_reduced)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

metrics = compute_classification_metrics(
    y_true=test_y_reduced,
    y_pred=test_preds,
    y_prob=test_probs
)

In [31]:
metrics_df = (
    pd.DataFrame([metrics])
    .drop(columns=["tp", "tn", "fp", "fn"])
    .round(4)
)

metrics_df

,accuracy,precision,recall,f1,auc,pr_auc
0,0.7928,0.6454,0.4866,0.5549,0.8484,0.6861


In [32]:
conf_matrix = pd.DataFrame(
    [[metrics["tn"], metrics["fp"]],
     [metrics["fn"], metrics["tp"]]],
    index=["Actual 0", "Actual 1"],
    columns=["Pred 0", "Pred 1"]
)

conf_matrix

,Pred 0,Pred 1
Actual 0,935,100
Actual 1,192,182


The final gradient boosting model demonstrates strong and consistent performance on the hold-out test set, indicating good generalization to unseen data.

The model achieves balanced classification performance, with an F1 score of **0.5549**, reflecting a stable trade-off between precision (**0.6454**) and recall (**0.4866**). This suggests that the model is effective at identifying churners while maintaining a controlled level of false positives.

From a ranking perspective, the model achieves a ROC-AUC of **0.8484** and a PR-AUC of **0.6861**, indicating strong ability to distinguish between churners and non-churners and to prioritize high-risk customers. These results are slightly improved compared to validation performance, further supporting the model’s robustness.

The confusion matrix shows that the model correctly identifies **182 churners** while missing **192**, and correctly classifies **935 non-churners** with **100 false positives**. This reflects a balanced classification behavior aligned with the chosen threshold and evaluation objectives.

Overall, the model provides reliable probability estimates and maintains stable performance across evaluation datasets. This supports its suitability for applications where customers are ranked by churn risk or targeted based on predicted likelihood of churn.

The consistency between cross-validation, validation, and test results indicates that the modeling approach and validation strategy were effective in controlling overfitting and selecting a robust final configuration.

## Executive Summary

This section presents the final results of the gradient boosting model developed for predicting customer churn.

The model builds on earlier stages of the workflow by leveraging a sequential ensemble of decision trees, where each tree iteratively improves upon the errors of the previous ones. This approach enables the model to capture complex non-linear relationships and interaction effects while maintaining strong predictive performance. Hyperparameters were tuned using cross-validation, and candidate models were evaluated on a separate validation set to ensure robust model selection.

A refined hyperparameter search was conducted around the initially selected configuration. However, this additional tuning resulted in only marginal performance improvements, indicating that the model had reached a point of diminishing returns. The originally selected model was therefore retained as the final configuration to preserve robustness and avoid overfitting to the validation set.

The final model was retrained on the combined training and validation data and evaluated on a hold-out test set to obtain an unbiased estimate of performance. The results indicate strong generalization, with consistent performance across cross-validation, validation, and test datasets.

On the test set, the model achieves balanced classification performance, with an F1 score of **0.5549**, reflecting a stable trade-off between precision (**0.6454**) and recall (**0.4866**). In addition, the model demonstrates strong ranking capability, with a ROC-AUC of **0.8484** and a PR-AUC of **0.6861**, indicating effective separation of churners from non-churners and reliable prioritization of high-risk customers.

Feature importance analysis shows that the model is primarily driven by **tenure**, **monthly charges**, and **contract type**, with additional contributions from service usage and payment-related features. This confirms that the model captures meaningful patterns related to customer lifecycle, pricing, and engagement.

Overall, the gradient boosting model provides a robust and high-performing predictive solution for churn estimation. Its ability to combine strong ranking performance with balanced classification metrics makes it well-suited for practical applications such as customer risk scoring and targeted retention strategies.